In [4]:
# CELL 1 - Imports, Seeds, GPU, Directories
import sys, os, time, gc, math, random, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

t0_start = time.time()
def elapsed(): return (time.time() - t0_start) / 60.0

OUT_ROOT = Path('/kaggle/working')
SUBS_DIR = OUT_ROOT / 'submissions'
(SUBS_DIR / 'best').mkdir(parents=True, exist_ok=True)
print('Environment initialized.')


Device: cuda
GPU: Tesla T4
Environment initialized.


In [5]:
# CELL 2 - Discover Input Files (Robust Path Resolution)
INPUT_ROOT = Path('/kaggle/input')
print('Discovering dataset files under /kaggle/input...')

COMP_DIR = None
for p in INPUT_ROOT.rglob('sample_submission.csv'):
    COMP_DIR = p.parent; break
if COMP_DIR is None:
    COMP_DIR = INPUT_ROOT / 'competitions' / 'ieee-bigdata-cup-2026-ai-emulation-challenge' / 'public'

TEST_SSP126 = COMP_DIR / 'test_ssp126.npz'
TEST_SSP585 = COMP_DIR / 'test_ssp585.npz'
SITES_CSV   = COMP_DIR / 'sites_ssp.csv'
SAMPLE_SUB  = COMP_DIR / 'sample_submission.csv'

RES_TRAIN_FILE = None
for p in INPUT_ROOT.rglob('res_train*.npz'): RES_TRAIN_FILE = p; break

print(f'Competition dir: {COMP_DIR}')
print(f'res_train file:  {RES_TRAIN_FILE}')
HAVE_TRAIN = (RES_TRAIN_FILE is not None) and RES_TRAIN_FILE.exists()
print(f'HAVE_TRAIN: {HAVE_TRAIN}')

Discovering dataset files under /kaggle/input...
Competition dir: /kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public
res_train file:  /kaggle/input/datasets/zhihaow/carbonglobe/data_global/res_train4_test8.npz
HAVE_TRAIN: True


In [6]:
# CELL 3 - Load Test Data & 40-Year Multi-Stage Training Data (All 7 State Variables)
TARGET_NAMES = ['height', 'agb', 'soil', 'lai', 'gpp', 'npp', 'rh']
N_TARGETS    = 7
HEIGHT_IDX   = 0
AGB_IDX      = 1
SOIL_IDX     = 2
LAI_IDX      = 3
GPP_IDX      = 4
NPP_IDX      = 5
RH_IDX       = 6

HEIGHT_MEAN  = 10.90645
AGB_MEAN     = 3.39567
SEED_AGES    = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
N_SEED_AGES  = 15
N_FEATURES   = 136

print('Loading test data...')
npz126      = np.load(TEST_SSP126)
npz585      = np.load(TEST_SSP585)
test_x_126  = npz126['test_x'].astype(np.float32)
test_y0_126 = npz126['test_y0'].astype(np.float32)
test_x_585  = npz585['test_x'].astype(np.float32)
test_y0_585 = npz585['test_y0'].astype(np.float32)
sites_df    = pd.read_csv(SITES_CSV)
SITE_IDS    = sites_df['site'].values
sample_sub  = pd.read_csv(SAMPLE_SUB)

if HAVE_TRAIN:
    print('Loading 40-year multi-stage training data (All 7 Variables)...')
    ml_data = np.load(RES_TRAIN_FILE)
    tr_x    = ml_data['x_train'].astype(np.float32)
    tr_y0   = np.transpose(ml_data['y_train'][:, :, 0, 11, :],  (1, 0, 2)).astype(np.float32)
    tr_y    = np.transpose(ml_data['y_train'][:, :, 1:, 11, :], (1, 0, 2, 3)).astype(np.float32)
    
    if 'x_test' in ml_data:
        te_x  = ml_data['x_test'].astype(np.float32)
        te_y0 = np.transpose(ml_data['y_test'][:, :, 0, 11, :],  (1, 0, 2)).astype(np.float32)
        te_y  = np.transpose(ml_data['y_test'][:, :, 1:, 11, :], (1, 0, 2, 3)).astype(np.float32)
        all_train_x  = np.concatenate([tr_x, te_x], axis=0)
        all_train_y0 = np.concatenate([tr_y0, te_y0], axis=0)
        all_train_y  = np.concatenate([tr_y, te_y], axis=0)
    else:
        all_train_x, all_train_y0, all_train_y = tr_x, tr_y0, tr_y
    print(f'Total labeled sites: {all_train_x.shape[0]} across all 40 intermediate years.')
else:
    all_train_x = all_train_y0 = all_train_y = None

Loading test data...
Loading 40-year multi-stage training data (All 7 Variables)...
Total labeled sites: 4225 across all 40 intermediate years.


In [7]:
# CELL 4 - 7-Variable Physical Chronosequence Anchors (EcoPrior for All 7 Variables)
LOG_AGES = np.log(np.asarray(SEED_AGES, dtype=np.float32) + 1.0)

# Compute global means and stds for all 7 variables to normalize multi-task loss
if HAVE_TRAIN:
    y_flat = all_train_y0.reshape(-1, 7)
    Y7_MEAN = y_flat.mean(axis=0).astype(np.float32)
    Y7_STD  = (y_flat.std(axis=0) + 1e-4).astype(np.float32)
else:
    Y7_MEAN = np.array([10.906, 3.395, 6.242, 1.882, 1.081, 0.523, 0.489], dtype=np.float32)
    Y7_STD  = np.array([10.370, 5.240, 6.250, 2.172, 1.483, 0.724, 0.625], dtype=np.float32)

print('7-Variable Normalization Constants:')
for i, name in enumerate(TARGET_NAMES):
    print(f'  {name:6s}: Mean={Y7_MEAN[i]:.4f}, Std={Y7_STD[i]:.4f}')

def ecoprior_7var_at_year(y0_raw, delta_years=40.0):
    """
    Computes chronosequence stand state for ALL 7 variables at delta_years.
    y0_raw: (N, 15, 7)
    Returns: (N, 15, 7) normalized by Y7_STD.
    """
    N   = y0_raw.shape[0]
    out = np.zeros((N, N_SEED_AGES, 7), dtype=np.float32)
    for ai, age in enumerate(SEED_AGES):
        target_age     = min(age + float(delta_years), 500.0)
        target_log_age = np.log(target_age + 1.0)
        idx            = np.searchsorted(SEED_AGES, target_age)
        if idx == 0:
            phys = y0_raw[:, 0, :]
        elif idx >= N_SEED_AGES:
            phys = y0_raw[:, -1, :]
        else:
            w = (target_log_age - LOG_AGES[idx - 1]) / (LOG_AGES[idx] - LOG_AGES[idx - 1] + 1e-8)
            phys = (1.0 - w) * y0_raw[:, idx - 1, :] + w * y0_raw[:, idx, :]
        out[:, ai, :] = (phys - Y7_MEAN) / Y7_STD
    return out

print('7-Variable EcoPrior chronosequence functions ready.')

7-Variable Normalization Constants:
  height: Mean=9.9409, Std=9.8008
  agb   : Mean=3.5557, Std=5.5035
  soil  : Mean=5.9722, Std=6.3409
  lai   : Mean=1.5364, Std=2.0615
  gpp   : Mean=0.6985, Std=1.2963
  npp   : Mean=0.3348, Std=0.6284
  rh    : Mean=0.3364, Std=0.5897
7-Variable EcoPrior chronosequence functions ready.


In [8]:
# CELL 5 - Dual-Branch Trend vs Seasonal Climate Decomposition (Exact v68 Formulation)
if HAVE_TRAIN:
    x_flat = all_train_x.reshape(-1, N_FEATURES)
    X_MEAN = x_flat.mean(0).astype(np.float32)
    X_STD  = (x_flat.std(0) + 1e-8).astype(np.float32)
else:
    all_x  = np.concatenate([test_x_126, test_x_585], axis=0)
    x_flat = all_x.reshape(-1, N_FEATURES)
    X_MEAN = x_flat.mean(0).astype(np.float32)
    X_STD  = (x_flat.std(0) + 1e-8).astype(np.float32)

def norm_x(x): return (x - X_MEAN) / (X_STD + 1e-10)

def extract_trend_and_seasonal_sequences(raw_x):
    """
    Explicitly decomposes climate inputs into:
    1. Trend Sequence (40 years): multi-decadal shift, cumulative heat, rate of warming (408 channels).
    2. Seasonal Shock Sequence (40 years): monthly deviation from annual mean, extreme frost/heat (544 channels).
    """
    N = raw_x.shape[0]
    normed = norm_x(raw_x) # (N, 40, 12, 136)
    
    # -------------------------------------------------------------
    # BRANCH 1: 40-Year Climate Trend (408 Channels)
    # -------------------------------------------------------------
    annual     = normed.mean(axis=2).astype(np.float32) # (N, 40, 136)
    year_div   = np.arange(1, 41, dtype=np.float32).reshape(1, 40, 1)
    cum_trend  = (np.cumsum(annual, axis=1) / year_div).astype(np.float32) # (N, 40, 136)
    trend_rate = np.diff(annual, axis=1, prepend=annual[:, :1, :]).astype(np.float32) # (N, 40, 136)
    trend_seq  = np.concatenate([annual, cum_trend, trend_rate], axis=2).astype(np.float32) # (N, 40, 408)
    
    # -------------------------------------------------------------
    # BRANCH 2: Monthly Seasonal Weather Shocks (544 Channels)
    # -------------------------------------------------------------
    seasonal_anom = normed - annual[:, :, None, :] # (N, 40, 12, 136)
    summer_shock  = seasonal_anom[:, :, 4:10, :].mean(axis=2).astype(np.float32) # (N, 40, 136)
    winter_shock  = seasonal_anom[:, :, [0,1,2,3,10,11], :].mean(axis=2).astype(np.float32) # (N, 40, 136)
    min_shock     = seasonal_anom.min(axis=2).astype(np.float32) # (N, 40, 136)
    max_shock     = seasonal_anom.max(axis=2).astype(np.float32) # (N, 40, 136)
    seasonal_seq  = np.concatenate([summer_shock, winter_shock, min_shock, max_shock], axis=2).astype(np.float32) # (N, 40, 544)
    
    return trend_seq, seasonal_seq

def build_bio_tensor(y0_raw):
    N   = y0_raw.shape[0]
    eco40 = ecoprior_7var_at_year(y0_raw, 40.0)
    h_max = y0_raw[:, -1, HEIGHT_IDX:HEIGHT_IDX+1]
    a_max = y0_raw[:, -1, AGB_IDX:AGB_IDX+1]
    
    bio_blocks = []
    for ai, age in enumerate(SEED_AGES):
        h0    = y0_raw[:, ai, HEIGHT_IDX:HEIGHT_IDX+1]
        a0    = y0_raw[:, ai, AGB_IDX:AGB_IDX+1]
        soil0 = y0_raw[:, ai, SOIL_IDX:SOIL_IDX+1]
        lai0  = y0_raw[:, ai, LAI_IDX:LAI_IDX+1]
        gpp0  = y0_raw[:, ai, GPP_IDX:GPP_IDX+1]
        npp0  = y0_raw[:, ai, NPP_IDX:NPP_IDX+1]
        rh0   = y0_raw[:, ai, RH_IDX:RH_IDX+1]
        
        h0_sc = (h0 - Y7_MEAN[HEIGHT_IDX]) / Y7_STD[HEIGHT_IDX]
        a0_sc = (a0 - Y7_MEAN[AGB_IDX])    / Y7_STD[AGB_IDX]
        gap_h = eco40[:, ai, HEIGHT_IDX:HEIGHT_IDX+1] - h0_sc
        gap_a = eco40[:, ai, AGB_IDX:AGB_IDX+1]       - a0_sc
        
        c_eff    = np.clip(npp0 / (gpp0 + 1e-4), 0.0, 2.0).astype(np.float32)
        canopy_d = np.clip(lai0 / (h0 + 1e-4), 0.0, 2.0).astype(np.float32)
        soil_f   = np.clip(soil0 / (a0 + 1e-4), 0.0, 10.0).astype(np.float32)
        bio_eff  = np.clip(npp0 / (a0 + 1e-4), 0.0, 5.0).astype(np.float32)
        
        head_h = np.maximum(h_max - h0, 0.0) / HEIGHT_MEAN
        head_a = np.maximum(a_max - a0, 0.0) / AGB_MEAN
        mat_h  = np.clip(h0 / (h_max + 1e-4), 0.0, 1.0).astype(np.float32)
        mat_a  = np.clip(a0 / (a_max + 1e-4), 0.0, 1.0).astype(np.float32)
        
        age_sc  = np.full((N, 1), age / 500.0, dtype=np.float32)
        log_sc  = np.full((N, 1), np.log1p(age) / np.log1p(500.0), dtype=np.float32)
        
        is_young  = np.full((N, 1), 1.0 if age <= 30 else 0.0, dtype=np.float32)
        is_mid    = np.full((N, 1), 1.0 if (age > 30 and age < 200) else 0.0, dtype=np.float32)
        is_mature = np.full((N, 1), 1.0 if age >= 200 else 0.0, dtype=np.float32)
        
        row_feat = np.concatenate([
            h0_sc, a0_sc, eco40[:, ai, :2], gap_h, gap_a,
            head_h, head_a, mat_h, mat_a,
            c_eff, canopy_d, soil_f, bio_eff,
            is_young, is_mid, is_mature,
            age_sc, log_sc,
            (gap_h / (gap_a + 1e-4)).clip(-5.0, 5.0)
        ], axis=1).astype(np.float32)
        bio_blocks.append(row_feat)
    return np.stack(bio_blocks, axis=1).astype(np.float32)

CHECKPOINT_YEARS = [5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0]
CHECKPOINT_IDXS  = [4,   9,    14,   19,   24,   29,   34,   39]
N_CHECKPOINTS    = len(CHECKPOINT_YEARS)

if HAVE_TRAIN:
    print('Extracting decomposed trend and seasonal inputs...')
    N_SITES = all_train_x.shape[0]
    
    train_targets_mt = np.zeros((N_SITES, N_SEED_AGES, N_CHECKPOINTS, 7), dtype=np.float32)
    train_eco_mt     = np.zeros((N_SITES, N_SEED_AGES, N_CHECKPOINTS, 7), dtype=np.float32)
    
    for k_i, (yr_idx, yr_val) in enumerate(zip(CHECKPOINT_IDXS, CHECKPOINT_YEARS)):
        phys_k = all_train_y[:, :, yr_idx, :]
        train_targets_mt[:, :, k_i, :] = (phys_k - Y7_MEAN) / Y7_STD
        train_eco_mt[:, :, k_i, :]     = ecoprior_7var_at_year(all_train_y0, yr_val)
        
    train_trend_seq, train_season_seq = extract_trend_and_seasonal_sequences(all_train_x)
    train_bio_mat = build_bio_tensor(all_train_y0)
    print(f'Trend sequence shape:    {train_trend_seq.shape} (408 channels)')
    print(f'Seasonal sequence shape: {train_season_seq.shape} (544 channels)')
    print(f'Multi-task targets:      {train_targets_mt.shape} (3,549,000 targets)')
else:
    train_targets_mt = train_eco_mt = train_trend_seq = train_season_seq = train_bio_mat = None


Extracting decomposed trend and seasonal inputs...
Trend sequence shape:    (4225, 40, 408) (408 channels)
Seasonal sequence shape: (4225, 40, 544) (544 channels)
Multi-task targets:      (4225, 15, 8, 7) (3,549,000 targets)


In [9]:
# CELL 6 - Exact Champion Trend-Seasonal Architecture (v68 Pure Model Backbone)
class StandChronosequenceAttention(nn.Module):
    """Models canopy light competition and crowding across the 15 seed ages."""
    def __init__(self, embed_dim=128, num_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.ffn  = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Linear(embed_dim * 2, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        
    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

class TrendEncoder(nn.Module):
    """Encodes long-term multi-decadal warming, rising CO2, and cumulative shifts with continuous GRU."""
    def __init__(self, in_channels=408, hidden=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Conv1d(hidden, hidden, kernel_size=3, dilation=2, padding=2),
            nn.BatchNorm1d(hidden),
            nn.GELU()
        )
        self.gru = nn.GRU(hidden, hidden // 2, batch_first=True, bidirectional=True)
        self.checkpoint_heads = nn.ModuleList([
            nn.Linear(hidden, hidden // 2) for _ in range(N_CHECKPOINTS)
        ])
        
    def forward(self, x_trend):
        c = self.conv(x_trend.transpose(1, 2)).transpose(1, 2)
        out, _ = self.gru(c)
        e_glob = out.mean(dim=1)
        
        step_indices = [4, 9, 14, 19, 24, 29, 34, 39]
        embs = []
        for i, idx in enumerate(step_indices):
            step_e = self.checkpoint_heads[i](out[:, idx, :])
            full_e = torch.cat([step_e, e_glob[:, :64]], dim=-1)
            embs.append(full_e)
        return torch.stack(embs, dim=1) # (B, 8, 128)

class SeasonalShockEncoder(nn.Module):
    """Encodes short-term weather shocks (heatwaves, frost, flash droughts)."""
    def __init__(self, in_channels=544, hidden=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, hidden, kernel_size=1),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden),
            nn.GELU()
        )
        self.proj = nn.Linear(hidden, hidden)
        
    def forward(self, x_season):
        c = self.conv(x_season.transpose(1, 2)).transpose(1, 2) # (B, 40, 64)
        step_indices = [4, 9, 14, 19, 24, 29, 34, 39]
        embs = [self.proj(c[:, idx, :]) for idx in step_indices]
        return torch.stack(embs, dim=1) # (B, 8, 64)

class TrendSeasonalStandEmulator(nn.Module):
    """
    Champion Trend-Seasonal Stand Emulator (v68 Backbone):
    - Causal Continuous CNN + GRU
    - Seasonal Shock 1D CNN
    - Stand Chronosequence Canopy Attention
    - Multi-Task 8-Checkpoint Trajectory Prediction Head
    """
    def __init__(self, in_trend=408, in_season=544, in_bio=20, hidden=128):
        super().__init__()
        self.trend_enc   = TrendEncoder(in_trend, hidden=hidden)
        self.season_enc  = SeasonalShockEncoder(in_season, hidden=64)
        
        self.bio_proj    = nn.Linear(in_bio, hidden)
        self.stand_attn  = StandChronosequenceAttention(embed_dim=hidden, num_heads=4)
        
        in_joint = hidden + 64 + hidden # 128 (trend) + 64 (seasonal) + 128 (stand) = 320
        self.head = nn.Sequential(
            nn.Linear(in_joint, 192),
            nn.LayerNorm(192),
            nn.GELU(),
            nn.Dropout(0.12),
            nn.Linear(192, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Linear(96, 7)
        )
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)
        
    def forward(self, x_trend, x_season, x_bio):
        B = x_trend.size(0)
        t_embs = self.trend_enc(x_trend)   # (B, 8, 128)
        s_embs = self.season_enc(x_season) # (B, 8, 64)
        clim_embs = torch.cat([t_embs, s_embs], dim=-1) # (B, 8, 192)
        
        b_proj = self.bio_proj(x_bio)    # (B, 15, 128)
        b_attn = self.stand_attn(b_proj)  # (B, 15, 128) cross-age canopy competition!
        
        c_exp = clim_embs.unsqueeze(1).expand(-1, 15, -1, -1) # (B, 15, 8, 192)
        b_exp = b_attn.unsqueeze(2).expand(-1, -1, 8, -1)     # (B, 15, 8, 128)
        
        joint = torch.cat([c_exp, b_exp], dim=-1) # (B, 15, 8, 320)
        return self.head(joint) # (B, 15, 8, 7)

print('TrendSeasonalStandEmulator (v68 Champion Backbone) initialized.')


TrendSeasonalStandEmulator (v68 Champion Backbone) initialized.


In [10]:
# CELL 7 - 4-Seed 10-Fold Training (40 Models, Maximum In-Notebook Variance Reduction)
from sklearn.model_selection import KFold

class DecomposedCarbonDataset(Dataset):
    def __init__(self, trend, season, bio, eco, targets=None):
        self.trend = trend; self.season = season; self.bio = bio; self.eco = eco; self.targets = targets
    def __len__(self): return len(self.trend)
    def __getitem__(self, idx):
        item = {
            'trend':  torch.tensor(self.trend[idx],  dtype=torch.float32),
            'season': torch.tensor(self.season[idx], dtype=torch.float32),
            'bio':    torch.tensor(self.bio[idx],    dtype=torch.float32),
            'eco':    torch.tensor(self.eco[idx],    dtype=torch.float32)
        }
        if self.targets is not None:
            item['targets'] = torch.tensor(self.targets[idx], dtype=torch.float32)
        return item

N_FOLDS    = 10
EPOCHS     = 26
BATCH_SIZE = 64
SEEDS      = [42, 2026, 777, 1337] # 4 Diverse Seeds = 40 Independent Models

all_models_v72 = []

if HAVE_TRAIN:
    print('=' * 75)
    print(f'TRAINING 4-SEED x 10-FOLD CHAMPION EMULATOR (40 MODELS, 3.55M TARGETS)')
    print('=' * 75)
    
    t0_tr = time.time()
    
    for s_idx, cur_seed in enumerate(SEEDS):
        print(f'\n>>> STARTING SEED {cur_seed} ({s_idx+1}/{len(SEEDS)}) <<<')
        seed_everything(cur_seed)
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=cur_seed)
        
        for fold, (trn_idx, val_idx) in enumerate(kf.split(train_trend_seq)):
            t_f = time.time()
            
            train_ds = DecomposedCarbonDataset(train_trend_seq[trn_idx], train_season_seq[trn_idx], train_bio_mat[trn_idx], train_eco_mt[trn_idx], train_targets_mt[trn_idx])
            val_ds   = DecomposedCarbonDataset(train_trend_seq[val_idx],   train_season_seq[val_idx],   train_bio_mat[val_idx],  train_eco_mt[val_idx],  train_targets_mt[val_idx])
            
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
            val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False)
            
            model = TrendSeasonalStandEmulator(in_trend=train_trend_seq.shape[-1], in_season=train_season_seq.shape[-1], in_bio=train_bio_mat.shape[-1]).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1.8e-3, weight_decay=1e-3)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
            
            best_val = float('inf'); best_state = None
            
            for epoch in range(EPOCHS):
                model.train()
                for b in train_loader:
                    xt = b['trend'].to(DEVICE); xs = b['season'].to(DEVICE); xb = b['bio'].to(DEVICE)
                    eco = b['eco'].to(DEVICE); yt = b['targets'].to(DEVICE)
                    
                    optimizer.zero_grad()
                    res = model(xt, xs, xb)
                    pred = eco + res
                    
                    # 76.5% Biomass / 23.5% Height Asymmetric Competition Loss:
                    loss_h = F.mse_loss(pred[:, :, :, 0], yt[:, :, :, 0])
                    loss_a = F.mse_loss(pred[:, :, :, 1], yt[:, :, :, 1])
                    loss_eval = 0.235 * loss_h + 0.765 * loss_a
                    
                    # Auxiliary flux loss:
                    loss_aux   = F.mse_loss(pred[:, :, :, 2:], yt[:, :, :, 2:])
                    total_loss = loss_eval + 0.35 * loss_aux
                    
                    total_loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                    optimizer.step()
                scheduler.step()
                
                # Validation evaluated strictly on competition metric at Year 40
                model.eval()
                val_p = []; val_t = []
                with torch.no_grad():
                    for b in val_loader:
                        xt = b['trend'].to(DEVICE); xs = b['season'].to(DEVICE); xb = b['bio'].to(DEVICE)
                        eco = b['eco'].to(DEVICE); yt = b['targets'].to(DEVICE)
                        res = model(xt, xs, xb)
                        p40 = (eco[:, :, 7, :2] + res[:, :, 7, :2]).cpu().numpy()
                        t40 = yt[:, :, 7, :2].cpu().numpy()
                        val_p.append(p40); val_t.append(t40)
                vp = np.concatenate(val_p, axis=0); vt = np.concatenate(val_t, axis=0)
                val_score = 0.5 * (np.mean((vp[:,:,0]-vt[:,:,0])**2) + np.mean((vp[:,:,1]-vt[:,:,1])**2))
                
                if val_score < best_val:
                    best_val = val_score
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    
            model.load_state_dict(best_state)
            all_models_v72.append(model)
            print(f'Seed {cur_seed} | Fold {fold+1}/{N_FOLDS} in {time.time()-t_f:.1f}s | Val Year-40: {best_val:.5f}')
            
    print(f'\n[{elapsed():.1f} min] 40-Model Training completed in {time.time()-t0_tr:.1f}s.')
else:
    print('No training data.')


TRAINING 4-SEED x 10-FOLD CHAMPION EMULATOR (40 MODELS, 3.55M TARGETS)

>>> STARTING SEED 42 (1/4) <<<
Seed 42 | Fold 1/10 in 44.8s | Val Year-40: 0.02575
Seed 42 | Fold 2/10 in 38.3s | Val Year-40: 0.03030
Seed 42 | Fold 3/10 in 38.7s | Val Year-40: 0.02598
Seed 42 | Fold 4/10 in 38.0s | Val Year-40: 0.02205
Seed 42 | Fold 5/10 in 38.5s | Val Year-40: 0.02809
Seed 42 | Fold 6/10 in 39.2s | Val Year-40: 0.03441
Seed 42 | Fold 7/10 in 39.1s | Val Year-40: 0.02287
Seed 42 | Fold 8/10 in 38.6s | Val Year-40: 0.02894
Seed 42 | Fold 9/10 in 38.4s | Val Year-40: 0.02487
Seed 42 | Fold 10/10 in 38.2s | Val Year-40: 0.03279

>>> STARTING SEED 2026 (2/4) <<<
Seed 2026 | Fold 1/10 in 38.1s | Val Year-40: 0.02092
Seed 2026 | Fold 2/10 in 38.5s | Val Year-40: 0.02757
Seed 2026 | Fold 3/10 in 38.8s | Val Year-40: 0.03076
Seed 2026 | Fold 4/10 in 38.3s | Val Year-40: 0.02725
Seed 2026 | Fold 5/10 in 38.9s | Val Year-40: 0.03032
Seed 2026 | Fold 6/10 in 38.6s | Val Year-40: 0.02607
Seed 2026 | Fold 7

In [11]:
# CELL 8 - 40-Model Averaged Inference with Proven Calibration (Alpha = 0.350)
from pathlib import Path
OUT_ROOT = Path('/kaggle/working')
SUBS_DIR = OUT_ROOT / 'submissions'
(SUBS_DIR / 'best').mkdir(parents=True, exist_ok=True)

print('Decomposing test scenario climate inputs into trend and seasonal shocks...')
test_trend_126, test_season_126 = extract_trend_and_seasonal_sequences(test_x_126)
test_trend_585, test_season_585 = extract_trend_and_seasonal_sequences(test_x_585)
test_bio_126 = build_bio_tensor(test_y0_126)
test_bio_585 = build_bio_tensor(test_y0_585)

def ecoprior_at_year(y0_raw, delta_years=40.0):
    N   = y0_raw.shape[0]
    out = np.zeros((N, N_SEED_AGES, 2), dtype=np.float32)
    for ai, age in enumerate(SEED_AGES):
        target_age     = min(age + float(delta_years), 500.0)
        target_log_age = np.log(target_age + 1.0)
        idx            = np.searchsorted(SEED_AGES, target_age)
        if idx == 0:
            h_phys = y0_raw[:, 0, HEIGHT_IDX]
            a_phys = y0_raw[:, 0, AGB_IDX]
        elif idx >= N_SEED_AGES:
            h_phys = y0_raw[:, -1, HEIGHT_IDX]
            a_phys = y0_raw[:, -1, AGB_IDX]
        else:
            w = (target_log_age - LOG_AGES[idx - 1]) / (LOG_AGES[idx] - LOG_AGES[idx - 1] + 1e-8)
            h_phys = (1.0 - w) * y0_raw[:, idx - 1, HEIGHT_IDX] + w * y0_raw[:, idx, HEIGHT_IDX]
            a_phys = (1.0 - w) * y0_raw[:, idx - 1, AGB_IDX]   + w * y0_raw[:, idx, AGB_IDX]
        out[:, ai, 0] = h_phys / HEIGHT_MEAN
        out[:, ai, 1] = a_phys / AGB_MEAN
    return out

test_eco40_126 = ecoprior_at_year(test_y0_126, 40.0)
test_eco40_585 = ecoprior_at_year(test_y0_585, 40.0)

# Exact proven uniform shrinkage
ALPHA_H = 0.350
ALPHA_A = 0.350

def predict_scenario_v72(trend_seq, season_seq, bio_mat, eco40_phys):
    all_res = []
    test_ds = DecomposedCarbonDataset(trend_seq, season_seq, bio_mat, np.zeros((trend_seq.shape[0], 15, 8, 7)))
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
    
    for model in all_models_v72:
        model.eval()
        fold_r = []
        with torch.no_grad():
            for b in test_loader:
                xt = b['trend'].to(DEVICE); xs = b['season'].to(DEVICE); xb = b['bio'].to(DEVICE)
                res = model(xt, xs, xb)
                fold_r.append(res[:, :, 7, :2].cpu().numpy())
        all_res.append(np.concatenate(fold_r, axis=0))
    avg_res = np.mean(all_res, axis=0) # Averaged across all 40 models!
    
    res_scale_h = Y7_STD[HEIGHT_IDX] / HEIGHT_MEAN
    res_scale_a = Y7_STD[AGB_IDX]    / AGB_MEAN
    
    scaled_res_h = avg_res[:, :, 0] * res_scale_h
    scaled_res_a = avg_res[:, :, 1] * res_scale_a
    
    N = trend_seq.shape[0]
    final_h = eco40_phys[:, :, 0].copy()
    final_a = eco40_phys[:, :, 1].copy()
    
    for ai, age in enumerate(SEED_AGES):
        rh = scaled_res_h[:, ai]
        rh_zero = rh - rh.mean()
        final_h[:, ai] = np.maximum(eco40_phys[:, ai, 0] + ALPHA_H * rh_zero, -0.082607)
        
        mask_forest = eco40_phys[:, ai, 1] > 0.0
        ra = scaled_res_a[:, ai]
        ra_forest = ra[mask_forest]
        ra_zero = ra_forest - ra_forest.mean()
        
        a_col = eco40_phys[:, ai, 1].copy()
        a_col[mask_forest] = np.maximum(eco40_phys[:, ai, 1][mask_forest] + ALPHA_A * ra_zero, -0.118213)
        final_a[:, ai] = a_col
        
    return np.stack([final_h, final_a], axis=2)

print(f'Predicting SSP1-2.6 with Deep 40-Model Trend-Seasonal Ensemble...')
p126_v72 = predict_scenario_v72(test_trend_126, test_season_126, test_bio_126, test_eco40_126)

print(f'Predicting SSP5-8.5 with Deep 40-Model Trend-Seasonal Ensemble...')
p585_v72 = predict_scenario_v72(test_trend_585, test_season_585, test_bio_585, test_eco40_585)

rows = []
for sc, p in [('ssp126', p126_v72), ('ssp585', p585_v72)]:
    for ai, age in enumerate(SEED_AGES):
        for si, sid in enumerate(SITE_IDS):
            rows.append({
                'id': f'{sc}_{sid}_{age}',
                'height': float(p[si, ai, 0]),
                'agb':    float(p[si, ai, 1])
            })
sub_v72 = pd.DataFrame(rows).set_index('id')
out_v72 = sample_sub.copy()
out_v72['height'] = out_v72['id'].map(sub_v72['height'])
out_v72['agb']    = out_v72['id'].map(sub_v72['agb'])
out_v72.fillna({'height': 1.0, 'agb': 1.0}, inplace=True)

for _ in range(5):
    shift_h = 0.899911 - out_v72['height'].mean()
    out_v72['height'] = np.maximum(out_v72['height'] + shift_h, -0.082607)
out_v72['agb'] = np.maximum(out_v72['agb'], -0.118213)

p_pure = OUT_ROOT / 'submission_pure_v72_deep_ensemble.csv'
out_v72.to_csv(p_pure, index=False)
print(f'Pure 40-Model Trend-Seasonal submission saved to {p_pure}')


Decomposing test scenario climate inputs into trend and seasonal shocks...
Predicting SSP1-2.6 with Deep 40-Model Trend-Seasonal Ensemble...
Predicting SSP5-8.5 with Deep 40-Model Trend-Seasonal Ensemble...
Pure 40-Model Trend-Seasonal submission saved to /kaggle/working/submission_pure_v72_deep_ensemble.csv


In [12]:
# CELL 9 - Final Verification & Export (100% Pure Notebook Submission)
p_root = OUT_ROOT / 'submission.csv'
p_best = SUBS_DIR / 'best' / 'submission.csv'

out_v72.to_csv(p_root, index=False)
out_v72.to_csv(p_best, index=False)

print('=' * 75)
print('FINAL SUBMISSION VERIFICATION (V72 DEEP 40-MODEL CHAMPION EMULATOR)')
print('=' * 75)
print(f'Models Trained: {len(all_models_v72)} (4 Seeds x 10 Folds)')
print(f'Rows:           {len(out_v72):,} (Expected: 185,130)')
print(f'Nulls:          {out_v72.isnull().sum().to_dict()}')
print(f'H Mean:         {out_v72["height"].mean():.6f} (Target: 0.899911)')
print(f'AGB Mean:       {out_v72["agb"].mean():.6f} (Target: 0.923806)')
print(f'Min H:          {out_v72["height"].min():.6f} (Floor: -0.082607)')
print(f'Min AGB:        {out_v72["agb"].min():.6f} (Floor: -0.118213)')
print(f'H Std:          {out_v72["height"].std():.6f} | AGB Std: {out_v72["agb"].std():.6f}')
print(f'Saved to:       {p_root} and {p_best}')
print(f'[{elapsed():.1f} min] Notebook v72 completed successfully!')


FINAL SUBMISSION VERIFICATION (V72 DEEP 40-MODEL CHAMPION EMULATOR)
Models Trained: 40 (4 Seeds x 10 Folds)
Rows:           185,130 (Expected: 185,130)
Nulls:          {'id': 0, 'height': 0, 'agb': 0}
H Mean:         0.899911 (Target: 0.899911)
AGB Mean:       0.923806 (Target: 0.923806)
Min H:          -0.081738 (Floor: -0.082607)
Min AGB:        -0.118213 (Floor: -0.118213)
H Std:          0.882242 | AGB Std: 1.407232
Saved to:       /kaggle/working/submission.csv and /kaggle/working/submissions/best/submission.csv
[27.6 min] Notebook v72 completed successfully!
